# 09 — Feature-Space Analysis: Audit Bertahap EFCIL × Attention

**Tidak menjalankan training.** Notebook ini tidak akan otomatis memuat 80 checkpoint. Alur wajib:

1. Audit schema prediction `.npz` (`keys`, `shape`, `dtype`, embedding ada/tidak).
2. Implementasikan adapter **spesifik tiap metode** berdasarkan *source code PyCIL asli*.
3. Jalankan satu **production checkpoint final task** untuk masing-masing FeTrIL, FeCAM, SSRE, PASS, EFC.
4. Gabungkan smoke audit PC/server; pastikan **lima metode PASSED**.
5. Ekstrak 20 baseline None (`baseline`) terlebih dahulu; jika aman, lanjut 80 run (`all`).
6. Analisis semua per-class dan per-pair feature geometry, confusion, dan perubahan attention.

**Penting:** `ready_snapshot` **bukan** bukti bobot checkpoint bisa dimuat. Model/loader asli tidak tersedia dalam notebook ini, sehingga adapter tidak boleh dikarang. Metrik AIA/LTA/FR/BWT existing tidak diubah.

In [ ]:
import sys, json, subprocess
from pathlib import Path
import pandas as pd
from IPython.display import display

CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD.parent if CWD.name == "notebooks" else CWD
SRC = PROJECT_ROOT / "scripts"
OUT = PROJECT_ROOT / "outputs" / "feature_analysis"
OUT.mkdir(parents=True, exist_ok=True)

# PC2: set 'pc2'; server: set 'server'. Folder local sources disesuaikan di bawah.
MACHINE = "pc2"  # pc2 | server | all
PC_SOURCE = "outputs/experiments"
SERVER_SOURCE = "analysis_inputs/server/experiments"
# Pada SERVER asli, set: MACHINE='server', SERVER_SOURCE='outputs/experiments'
ADAPTER = "scripts/feature_adapter_template.py"
MODE = "baseline"  # setelah baseline lengkap, ganti ke all
RUN_SMOKE = False    # ubah ke True hanya setelah adapter asli selesai
RUN_EXTRACTION = False
RUN_ANALYSIS = False
GLOBAL_SMOKE_SUMMARY = None   # e.g. outputs/feature_analysis/global_smoke_summary.json

print("Root:", PROJECT_ROOT)
print("Machine:", MACHINE)
print("Adapter:", PROJECT_ROOT/ADAPTER)


## 1. Audit prediction `.npz` (langsung dijalankan)

Periksa apakah `predictions/task_XX.npz` sudah memuat embedding. Jangan mengasumsikannya tersedia.

In [ ]:
cmd = [sys.executable, str(SRC/"feature_space_pipeline.py"), "inspect-npz",
       "--project-root", str(PROJECT_ROOT), "--pc-source", PC_SOURCE,
       "--server-source", SERVER_SOURCE, "--machine", MACHINE]
subprocess.run(cmd, check=True)
npz_audit = pd.read_csv(OUT/"prediction_npz_audit.csv")
display(npz_audit[["run_id", "method", "prediction_path", "status", "has_embeddings", "arrays"]].head(10))
print("NPZ yang menyimpan embedding:", int(npz_audit.has_embeddings.sum()) if len(npz_audit) else 0)


## 2. Production checkpoint smoke — satu per metode pada mesin pemilik bobot

**Jangan aktifkan `RUN_SMOKE` sebelum `scripts/feature_adapter_template.py` diimplementasikan berdasarkan learner asli.**

Di mesin yang checkpoint-nya tersedia, jalankan smoke. Jika masing-masing metode berada di mesin berbeda, gabungkan `feature_checkpoint_audit.csv` menggunakan `scripts/merge_feature_smoke.py`. Tidak ada ekstraksi batch sebelum lima metode lolos secara global dan metode lokal lolos pada mesin masing-masing.

In [ ]:
if RUN_SMOKE:
    cmd = [sys.executable, str(SRC/"feature_space_pipeline.py"), "smoke",
           "--project-root", str(PROJECT_ROOT), "--pc-source", PC_SOURCE,
           "--server-source", SERVER_SOURCE, "--machine", MACHINE,
           "--adapter", ADAPTER, "--smoke-samples", "16"]
    subprocess.run(cmd, check=True)
    audit = pd.read_csv(OUT/"feature_checkpoint_audit.csv")
    display(audit)
else:
    print("Smoke tidak dijalankan otomatis. Implementasikan adapter asli dan aktifkan RUN_SMOKE.")


## 3. Batch feature extraction (terkunci sampai smoke lulus)

Jalankan pada mesin yang menyimpan bobot asli, lalu kumpulkan hasil `feature_embeddings/*.npz` dari PC/server pada satu folder analisis. Ekstraksi baseline dahulu, baru mode all. `y_true` harus cocok dengan canonical prediction (urutan sama), feature harus 512-d dan finite, jumlah test AID final 2.000.

In [ ]:
if RUN_EXTRACTION:
    cmd = [sys.executable, str(SRC/"feature_space_pipeline.py"), "extract",
           "--project-root", str(PROJECT_ROOT), "--pc-source", PC_SOURCE,
           "--server-source", SERVER_SOURCE, "--machine", MACHINE,
           "--adapter", ADAPTER, "--mode", MODE]
    if GLOBAL_SMOKE_SUMMARY:
        cmd += ["--global-smoke-summary", GLOBAL_SMOKE_SUMMARY]
    subprocess.run(cmd, check=True)
    display(pd.read_csv(OUT/"feature_extraction_audit.csv"))
else:
    print("Batch extraction tidak dijalankan otomatis.")


## 4. Analisis fitur + audit keluaran

Setelah embedding valid terkumpul, script menghasilkan similarity, dispersion, confusion join, Spearman deskriptif, perubahan attention, pasangan kelas terpilih dan CSV siap Bab IV. Grafik untuk run pertama dibuat dengan 300 dpi.

In [ ]:
if RUN_ANALYSIS:
    subprocess.run([sys.executable, str(SRC/"analyze_feature_space.py"),
                    "--project-root", str(PROJECT_ROOT), "--plot-max-runs", "5"], check=True)
    summary = json.loads((OUT/"feature_analysis_manifest.json").read_text(encoding="utf-8"))
    print(json.dumps(summary, indent=2, ensure_ascii=False))
    for fn in ["inter_class_similarity_top_pairs.csv", "intra_class_dispersion.csv",
               "feature_correlation_summary.csv", "attention_feature_summary.csv",
               "feature_analysis_errors.csv"]:
        p=OUT/fn
        if p.exists() and p.stat().st_size>0:
            try:
                print(fn)
                display(pd.read_csv(p).head(10))
            except pd.errors.EmptyDataError:
                print("No observations yet")
else:
    print("Analisis hanya dijalankan setelah embedding production tersedia dan tervalidasi.")


## 5. Interpretasi dan batasan

- Cosine similarity mengukur kedekatan **centroid fitur model**, bukan kemiripan visual absolut kelas.
- Cosine distance / delta similarity **tidak** memakai percentage points.
- Confusion rate dan delta accuracy memakai **persen/percentage points** secara konsisten.
- Cek pasangan negatif/degradasi juga; jangan pilih hanya contoh sukses attention.
- Korelasi Spearman bersifat deskriptif karena hanya satu seed dan class pairs tidak independen.
- Jika tidak ada `sample_path` yang valid, contact sheet tidak dapat dibuat. Jangan mengganti dengan citra acak.
